# Step 1: Data collection

# Step 2: Data Preprocessing


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
# Define paths
ANNOTATIONS_DIR = "/content/drive/MyDrive/internProject/annotations"
IMAGES_DIR = "/content/drive/MyDrive/internProject/images"
OUTPUT_DIR = "outputs"

In [3]:
# Create necessary directories
import os

os.makedirs(os.path.join(OUTPUT_DIR, 'images', 'train'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'images', 'val'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'labels', 'train'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'labels', 'val'), exist_ok=True)

In [ ]:
# reading all XML annotation files in a folder, 
# extracts each image’s filename and the objects in it (with class names and bounding box coordinates), and stores everything in a list.
import xml.etree.ElementTree as ET

annotations = []
for xml_file in os.listdir(ANNOTATIONS_DIR):
    tree = ET.parse(os.path.join(ANNOTATIONS_DIR, xml_file))
    root = tree.getroot()
    file_name = root.find('filename').text
    objects = []
    for obj in root.findall('object'):
        class_name = obj.find('name').text
        bndbox = obj.find('bndbox')
        xmin = int(bndbox.find('xmin').text)
        ymin = int(bndbox.find('ymin').text)
        xmax = int(bndbox.find('xmax').text)
        ymax = int(bndbox.find('ymax').text)
        objects.append((class_name, xmin, ymin, xmax, ymax))
    annotations.append((file_name, objects))

In [5]:
# Split dataset into training and validation sets
train_annotations = annotations[:int(0.8*len(annotations))]
val_annotations = annotations[int(0.8*len(annotations)):]

In [6]:
# Encode labels
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
all_labels = [obj[0] for ann in annotations for obj in ann[1]]
label_encoder.fit(all_labels)

LabelEncoder()

In [ ]:
# Helper function to convert annotations(bounding box) to YOLO format
def convert_to_yolo_format(size, box):
    dw = 1./size[0]
    dh = 1./size[1]
    x = (box[0] + box[2]) / 2.0
    y = (box[1] + box[3]) / 2.0
    w = box[2] - box[0]
    h = box[3] - box[1]
    x = x*dw
    w = w*dw
    y = y*dh
    h = h*dh
    return (x,y,w,h)

In [ ]:
# Save images and labels in YOLO format
import shutil
import cv2

def save_yolo_files(annotations, image_dir, label_dir):
    for file_name, objects in annotations:
        img_path = os.path.join(IMAGES_DIR, file_name)
        img = cv2.imread(img_path)
        h, w, _ = img.shape
        shutil.copy(img_path, image_dir)
        label_path = os.path.join(label_dir, file_name.replace('.png', '.txt'))
        with open(label_path, 'w') as f:
            for obj in objects:
                class_name, xmin, ymin, xmax, ymax = obj
                class_id = label_encoder.transform([class_name])[0]
                bb = convert_to_yolo_format((w, h), (xmin, ymin, xmax, ymax))
                f.write(f"{class_id} {' '.join(map(str, bb))}\n")

# Save train and val data
save_yolo_files(train_annotations, os.path.join(OUTPUT_DIR, 'images', 'train'), os.path.join(OUTPUT_DIR, 'labels', 'train'))
save_yolo_files(val_annotations, os.path.join(OUTPUT_DIR, 'images', 'val'), os.path.join(OUTPUT_DIR, 'labels', 'val'))

In [ ]:
# Create data.yaml file: tells YOLO where the dataset is located and how to interpret the labels during training.
data_yaml = f"""
train: {os.path.join(OUTPUT_DIR, 'images', 'train')}
val: {os.path.join(OUTPUT_DIR, 'images', 'val')}

nc: {len(label_encoder.classes_)}
names: {label_encoder.classes_.tolist()}
"""

with open(os.path.join(OUTPUT_DIR, 'data.yaml'), 'w') as f:
    f.write(data_yaml)